[Reference](https://blog.stackademic.com/how-to-fine-tune-llms-with-lora-qlora-26953ef326f3?sk=ac8354362ab1efa7ced2335f9f4c7fdf$0)

In [1]:
!pip install torch transformers peft datasets accelerators

ERROR: Could not find a version that satisfies the requirement accelerators (from versions: none)
ERROR: No matching distribution found for accelerators


In [3]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 7.6 MB/s  0:00:00


# Low-Rank Adaptation (LoRA)

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
from trl import SFTTrainer
import torch
from huggingface_hub import login

HF_TOKEN = None

# Kaggle secret support
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF_TOKEN from Kaggle secrets.")
except Exception:
    pass

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()

# ============ TRAINING SECTION ============

# Load a pre-trained model
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Configure LoRA
lora_config = LoraConfig(
    r=8,                           # Rank
    lora_alpha=16,                 # Scaling factor
    lora_dropout=0.1,              # Dropout for LoRA layers
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]  # Which layers to adapt
)

# Prepare your dataset
dataset = load_dataset("yahma/alpaca-cleaned", split="train").select(range(500))
def format_instruction(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"}
dataset = dataset.map(format_instruction)

# Trainer setup
training_args = TrainingArguments(
    output_dir="./smol-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    report_to="none",
    save_strategy="no"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config
)

# Train
trainer.train()
print("Saving model...")
trainer.model.save_pretrained("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")
print("Model saved to ./lora_adapter")

# ============ INFERENCE SECTION ============

# First, reload the base model
base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_tokenizer = AutoTokenizer.from_pretrained(model_name)
base_tokenizer.pad_token = base_tokenizer.eos_token

trained_model = PeftModel.from_pretrained(base_model, "./lora_adapter")
trained_model.eval()  # Set to evaluation mode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = trained_model.to(device)

def generate_response(instruction, max_new_tokens=100, temperature=0.7):
    """Generate a response from the fine-tuned model"""
    # Format prompt the same way as training
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    # Tokenize
    inputs = base_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate
    with torch.no_grad():
        outputs = trained_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.95,
            pad_token_id=base_tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    # Decode and extract response
    full_response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the response part (after "### Response:\n")
    if "### Response:\n" in full_response:
        response = full_response.split("### Response:\n")[-1]
    else:
        response = full_response
    return response.strip()

# Test the model with sample questions
test_questions = [
    "What is machine learning?",
    "Explain quantum computing in simple terms",
    "How do I learn Python programming?",
    "What are the benefits of exercise?",
    "Tell me a fun fact about space."
]

print("\n" + "="*60)
print("SAMPLE RESPONSES")
print("="*60)

for i, question in enumerate(test_questions, 1):
    print(f"\nQ{i}: {question}")
    print(f"A: {generate_response(question)}")
    print("-" * 50)

# Quantized Low-Rank Adaptation (QLoRA)


In [2]:
!pip install "transformers>=4.44.0" "datasets>=2.20.0" "peft>=0.12.0" "bitsandbytes>=0.44.0" "trl>=0.9.6" "accelerate>=0.33.0" scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 55.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 41.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [trl]


In [3]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, PeftModel, TaskType
from trl import SFTTrainer

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "./qlora-tinyllama-alpaca/final-adapter"

# Load the dataset
dataset = load_dataset("tatsu-lab/alpaca", split="train")
def format_prompt(example):
    if example["input"]:
        return {"text": f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"}
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"}
dataset = dataset.map(format_prompt).select(range(2000))

# QLoRA Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Load model weights in 4-bit
    bnb_4bit_quant_type="nf4",      # NF4: optimal for normally distributed weights
    bnb_4bit_compute_dtype=torch.bfloat16,  # Use fp16 for actual computations
    bnb_4bit_use_double_quant=True, # Double quantization: quantize the quantization constants too
)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

# Configure LoRA Adapters
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Training Configuration
training_args = TrainingArguments(
    output_dir="./qlora-out",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    bf16=True,       # ← use this
    fp16=False,      # ← not this
    logging_steps=25,
    save_strategy="epoch",
    report_to="none",
    warmup_steps=50,
)

# Train the model
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
    formatting_func=lambda x: x["text"],
)
trainer.train()

# Save only the LoRA adapter weights
trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

# ======================= INFERENCING ===========================

def generate_response(instruction, input_text="", max_new_tokens=200):
    """
    Generate a response from the fine-tuned model.

    Args:
        instruction : The task instruction
        input_text  : Optional context (can be empty)
        max_new_tokens: Maximum tokens to generate
    """
    # Format prompt (same template as training)
    if input_text:
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )
    else:
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n"
        )

    # Tokenize
    inputs = ft_tokenizer(prompt, return_tensors="pt").to(ft_model.device)
    input_len = inputs["input_ids"].shape[1]

    # Generate
    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,        # Controls randomness (lower = more deterministic)
            top_p=0.9,              # Nucleus sampling
            do_sample=True,
            repetition_penalty=1.1, # Penalize repeated tokens
            pad_token_id=ft_tokenizer.eos_token_id,
        )

    # Decode only the new tokens (skip the input prompt)
    response = ft_tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )
    return response.strip()

# Reload base model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load the LoRA adapter on top
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()  # Switch to eval mode

# Load tokenizer
ft_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

instruction = "Explain what machine learning is in simple terms."
response = generate_response(instruction)
print(f"RESPONSE:\n{response}")